In [ ]:
import sys
import os
import os.path as osp

import numpy as np
import torch
import trimesh
import cv2
import open3d as o3d

from scipy.spatial.distance import cdist
from transforms3d.axangles import axangle2mat, mat2axangle
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from mano_pybullet.hand_model import HandModel20

In [2]:
sys.path.append("..")

from model.hand_opt import AdamGraspCmap

from utils.grasp_utils import (
    get_handmodel,
    regularize_pc_point_count,
    convert_4x4_to_7dpose,
    convert_7dpose_to_4x4,
    convert_aligned_to_gripper_pose,
    convert_gripper_to_aligned_pose,
)

from utils.pc_utils import (
    backproject_camera, 
    compute_xyz, 
    load_depth_img,
    filter_outliers
)

from utils.fig_utils import (
    plot_point_cloud,
    plot_point_cloud_cmap,
    plot_trimesh_mesh
)

In [3]:
def apply_extrinsics(points, RT_camera):
    """
    Transform the point cloud using the camera extrinsics.
    
    Args:
        points: (N,3) numpy array.
        RT_camera: (4,4) tf (rotation and translation) for camera
    
    Returns:
        Transformed Open3D PointCloud object.
    """
    rotation = RT_camera[:3, :3]
    translation = RT_camera[:3, 3]
    transformed_points = (rotation @ points.T).T + translation.T
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(transformed_points)
    return pcd

def estimate_normals_with_open3d(point_cloud, camera_position=None):
    """
    Estimate normals of a point cloud and orient them.
    
    Args:
        point_cloud: Open3D PointCloud object.
        camera_position: Optional 3x1 numpy array of camera position for normal orientation.
    
    Returns:
        Point cloud with estimated normals.
    """
    # Estimate normals
    point_cloud.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    
    # Orient normals towards the camera or in a consistent direction
    if camera_position is not None:
        point_cloud.orient_normals_towards_camera_location(camera_position)
    else:
        point_cloud.orient_normals_consistent_tangent_plane(k=30)
    
    return point_cloud

def transform_to_camera_frame(point_cloud, RT_camera):
    """
    Transform both points and normals from the world frame to the camera frame.

    Args:
        point_cloud: Open3D PointCloud object with estimated normals.
        rotation: 3x3 numpy array representing the rotation matrix of the camera extrinsics.
        translation: 3x1 numpy array representing the translation vector of the camera extrinsics.

    Returns:
        Open3D PointCloud object with points and normals in the camera frame.
    """
    # Get points and normals from the point cloud
    points = np.asarray(point_cloud.points)
    normals = np.asarray(point_cloud.normals)

    rotation = RT_camera[:3, :3]
    translation = RT_camera[:3, 3]
    translation = translation.reshape(1, 3)

    # Compute inverse rotation matrix
    rotation_inverse = rotation.T  # Inverse of rotation (R^-1)

    # Transform points to the camera frame
    transformed_points = (rotation_inverse @ (points - translation).T).T

    # Transform normals to the camera frame
    transformed_normals = (rotation_inverse @ normals.T).T

    # Update the point cloud with transformed points and normals
    tf_pcd = o3d.geometry.PointCloud()
    tf_pcd.points = o3d.utility.Vector3dVector(transformed_points)
    tf_pcd.normals = o3d.utility.Vector3dVector(transformed_normals)
    return tf_pcd

In [4]:
def compute_contact_map(gripper_pts, obj_pts):
    pairwise_distances = cdist(obj_pts, gripper_pts)
    distances = np.min(pairwise_distances, axis=1)
    contact_m = 2 / (1 + np.exp(distances))
    return contact_m


def compute_contact_map_aligned(gripper_pts, obj_pts, obj_normals):
    pairwise_distances = cdist(obj_pts, gripper_pts)

    pairwise_differences = obj_pts[:, np.newaxis] - gripper_pts
    pairwise_alignments = np.sum(
        pairwise_differences * obj_normals[:, np.newaxis, :], axis=-1
    )
    pairwise_alignments = np.exp(1 - pairwise_alignments)

    distances = np.min(pairwise_distances * pairwise_alignments, axis=1)
    contact_m = 2 / (1 + np.exp(distances))
    return contact_m


# Constants

In [5]:
FIG_SAVE_DIR = "./viz_debug/"
GLOBAL_SEED = 42
THRESHOLD_DIST_LOCAL = 0.08
DEVICE = 'cuda' # 'cpu'

# Gripper Model and CamK

In [6]:
target_gripper = "fetch_gripper"
device = DEVICE

target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [7]:
fetch_gripper_mesh = trimesh.load("../data/fetch_gripper_base_pose.obj")

In [ ]:
# NEW CAMERA INTRINSICS
k = [527.8869068647631, 0.0, 321.7148665756361, 0.0, 524.7942507494529, 230.2819198622499, 0.0, 0.0, 1.0]
intrinsics = np.array(k).reshape(3, 3)
fx = intrinsics[0, 0]
fy = intrinsics[1, 1]
px = intrinsics[0, 2]
py = intrinsics[1, 2]
print(intrinsics)

# Set Sample Data

In [ ]:
TASKS_DIR = "/home/ninad/Datasets/MMDemo/newCamK"
task_names = sorted(os.listdir(TASKS_DIR))
print(task_names)
TO_SAVE = False

In [ ]:
task_id = task_names[2]
print("TASK:", task_id)
input_dir = osp.join(TASKS_DIR, task_id)

# CONSIDERING the "Task 18 Move Chair" and Frame 31 from it
frame_id = "000031"
print("FRAME:", frame_id)

In [ ]:
pose_log_dir = osp.join(input_dir, "pose")
hamer_root_dir = osp.join(input_dir, "out", "hamer")
hamer_npz_dir = osp.join(hamer_root_dir, "model")
depth_img_dir = osp.join(input_dir, "depth")

samv2_dir = osp.join(input_dir, "out", "samv2")
# NOTE: Assuming only 1 folder within the samv2 directory
masks_dir = osp.join(samv2_dir, os.listdir(samv2_dir)[0], "obj_masks")
print(masks_dir) 

npz_files = [
    f
    for f in os.listdir(hamer_npz_dir)
    if osp.isfile(osp.join(hamer_npz_dir, f)) and f.lower().endswith((".npz"))
]
if not npz_files:
    raise ValueError(f"No npz files found in {hamer_npz_dir}!....")

npz_files = sorted(npz_files)
print(npz_files)

In [ ]:
# LOAD the npz data for the frame

npzfname = ""
for f in npz_files:
    fid = osp.splitext(f)[0]
    if fid == frame_id:
        npzfname = f
        print("Found the npzfname!:", f)
        break
print(npzfname)


npz_fpath = osp.join(hamer_npz_dir, npzfname)
npz_data = dict(
    np.load(npz_fpath, allow_pickle=True)
)  # load the npz as dict to be able to update later
RT_grippers = npz_data["target_transfer_pose"]

In [ ]:
print(RT_grippers.shape)
RT_old = RT_grippers[0]
print(RT_old)

In [ ]:
for k in npz_data.keys():
    print(k)

In [ ]:
# LOAD THE CAMERA POSE
pose_data = dict(
    np.load(osp.join(pose_log_dir, npzfname))
)
RT_camera = pose_data['RT_camera']
print(RT_camera)

In [ ]:
pose_data

## Load Obj PC from first view

In [ ]:
# LOAD the object PC using the initial view

# Keeping the first frame id as 1 instead of 0
first_frame_id = "000001"
depth_img_f = osp.join(depth_img_dir, f"{first_frame_id}.png")
depth_im = load_depth_img(depth_img_f)

mask_f = osp.join(masks_dir, f"{first_frame_id}.png")
mask_im = cv2.imread(mask_f, 0)

# scene_pc = compute_xyz(depth_im, fx, fy, px, py)
obj_pc_first_view = backproject_camera(depth_im, intrinsics, target_mask=mask_im)
print(obj_pc_first_view.shape)

In [18]:
# # VISUALIZE the object PC

# vis_data = []
# vis_data += [
#     plot_point_cloud(obj_pc_first_view)
# ]
# fig = go.Figure(data=vis_data)
# fig.show()


In [19]:
# vis_data = []
# vis_data += [
#     plot_trimesh_mesh(fetch_gripper_mesh.copy().apply_transform(RT_old))
# ]
# fig = go.Figure(data=vis_data)
# fig.show()



In [20]:
# Combined VIZ
# vis_data = []

# vis_data += [
#     plot_trimesh_mesh(fetch_gripper_mesh.copy().apply_transform(RT_old), color='red', opacity=1)
# ]

# vis_data += [
#     plot_point_cloud(obj_pc_first_view)
# ]

# fig = go.Figure(data=vis_data)
# # fig.show()
# fig.write_html(osp.join(FIG_SAVE_DIR, "original.html"))

## Perturb Grasp Pose

In [21]:
# Perturb the gripper pose
# We move 5cm (0.05m) forward in the gripper palm normal direction
RT_collision = RT_old.copy()
delta = 0.05 
rx = RT_old[:3, 0] # x comp for rotn mat
t_old = RT_old[:3, 3]
t_new = t_old + delta * rx # note the element wise multiplication
# also eqivalent: t_new = t_old + np.dot(RT_old[:3, :3], delta) # R * delta + t_old
RT_collision[:3, 3] = t_new

In [22]:
# # Always make a copy and apply the transform!

# vis_data = []
# vis_data += [
#     plot_trimesh_mesh(fetch_gripper_mesh.copy().apply_transform(RT_collision), color='red', opacity=1)
# ]
# vis_data += [
#     plot_point_cloud(obj_pc_first_view)
# ]

# fig = go.Figure(data=vis_data)
# # fig.show()
# fig.write_html(osp.join(FIG_SAVE_DIR, "collision.html"))

# GraspOpt Fix for Perturbed Pose

## Local Region of ObjPC 

In [ ]:
# NOTE
# First determine the local region for the object pc using distance to fetch gripper mesh points

gripper_pts = np.array(
    trimesh.sample.sample_surface_even(
            mesh=fetch_gripper_mesh.copy().apply_transform(RT_collision),
            count=256,
            seed=GLOBAL_SEED,
        )[0]
)
obj_hand_dist = np.min(cdist(obj_pc_first_view, gripper_pts), axis=1)
print("Min, Max dist", np.min(obj_hand_dist), np.max(obj_hand_dist))

idxs_close = obj_hand_dist < THRESHOLD_DIST_LOCAL
obj_pc_subset = obj_pc_first_view[idxs_close]
print("Subset objpc shape:", obj_pc_subset.shape)

# DO FPS on selected object points
pts = regularize_pc_point_count(obj_pc_subset, 1000, use_farthest_point=True)[0]
obj_pc_subset = pts


In [24]:
# ## VIZ: Local Obj PC region and Gripper Pts used for determination

# vis_data = []
# # vis_data += [
# #     plot_trimesh_mesh(fetch_gripper_mesh.copy().apply_transform(RT_collision), color='red', opacity=1)
# # ]
# # vis_data += [
# #     plot_point_cloud(obj_pc_first_view, size=3)
# # ]
# vis_data += [
#     plot_point_cloud(obj_pc_subset, color='blue', size=3)
# ]
# vis_data += [
#     plot_point_cloud(gripper_pts, color='red', size=3)
# ]


# fig = go.Figure(data=vis_data)
# fig.show()


## Object PC Normals

In [25]:
objpcd_with_normals_in_world = estimate_normals_with_open3d(
    apply_extrinsics(obj_pc_subset, RT_camera),
    RT_camera[:3, 3]
)

# Get in camera frame
objpcd_with_normals_in_camera = transform_to_camera_frame(
    objpcd_with_normals_in_world,
    RT_camera
)

### visualize
# o3d.visualization.draw_geometries([objpcd_with_normals_in_camera], point_show_normal=True)

assert np.allclose(np.asarray(objpcd_with_normals_in_camera.points), obj_pc_subset)

objpc_pts = np.array(obj_pc_subset) # should be equal to objpcs_with_normals_in_camera.points
objpc_nrm = np.asarray(objpcd_with_normals_in_camera.normals)


In [ ]:
contact_map = compute_contact_map(gripper_pts, objpc_pts)
# contact_map = compute_contact_map_aligned(gripper_pts, objpc_pts, objpc_nrm)
print(contact_map.shape, objpc_pts.shape, objpc_nrm.shape)

In [27]:
# ## VIZ: Local Obj PC region and Gripper Pts + Contact Map

# vis_data = []

# vis_data += [
#     plot_point_cloud_cmap(objpc_pts, color_levels=contact_map, size=3)
# ]
# vis_data += [
#     plot_point_cloud(gripper_pts, color='gray', size=3, opacity=0.3)
# ]

# fig = go.Figure(data=vis_data)
# fig.show()


## Grasp Opt Init

In [ ]:
### Construct the contact map goal: 
# # Goal = [obj pc points (N,3), obj pc normals (N,3), contact map (N, 1)] 
# # Shape = (N, 7)

cmap_goal = np.concatenate([objpc_pts, objpc_nrm, contact_map.reshape(-1, 1)], axis=1)
print(cmap_goal.shape)
cmap_tensor = torch.tensor(cmap_goal)

In [ ]:
## The initial "colliding" grasp will the Source Grasp

source_q = torch.zeros(9)
source_q[:3] = torch.tensor(RT_collision[:3, 3])
source_q[3:] = torch.tensor(RT_collision.T.reshape(-1)[:6])
source_q = source_q.to(DEVICE)
if source_q.shape[0] != 9 + len(target_model.dynamic_joints):
  # We optimized only for pose, so need to provide dummy joints
  source_q = torch.cat((source_q, (target_model.dynamic_joints_q_upper[0] - target_model.dynamic_joints_q_mid[0])), dim=0)

print(source_q)

In [ ]:
grasp_transfer_opt = AdamGraspCmap(
  target_robot_name=target_gripper,
  source_grasp=source_q,
  learning_rate=1e-3,
  device=DEVICE,
  collision_weight=2,
)

q_traj, energy, _ = grasp_transfer_opt.run_adam(
    contact_map_goal=cmap_tensor,
    source_grasp = source_q,
    running_name="test"
)


In [ ]:
min_energy_index = energy.min(dim=0)[1]
print(min_energy_index.item())

print(q_traj.shape)
best_q = q_traj[min_energy_index.item(), -1]
print(best_q.shape)

if best_q.shape[0] != 9 + len(target_model.dynamic_joints):
  # We optimized only for pose, so need to provide dummy joints
  best_q = torch.cat(
        (
          best_q, 
          (target_model.dynamic_joints_q_upper[0] - target_model.dynamic_joints_q_mid[0])
        ), 
        dim=0
    )

## GrasOpt Result Viz

In [35]:
# VIZ

target_gripper_mesh_data = target_model.get_plotly_data(
    q=best_q.unsqueeze(0).float().to(device), 
    color='orange', 
    opacity=0.6
)

## VIZ: Local Obj PC region and Gripper Pts + Contact Map

vis_data = []

vis_data += [
    plot_point_cloud_cmap(objpc_pts, color_levels=contact_map, size=3)
]
vis_data += [
    plot_point_cloud(gripper_pts, color='darkgray', size=3, opacity=0.6)
]
vis_data += target_gripper_mesh_data

fig = go.Figure(data=vis_data)
# fig.show()
fig.write_html(osp.join(FIG_SAVE_DIR, "result.html"))
